In [1]:
import os
import glob
import pandas as pd
from pathlib import Path

# 1. Find where this notebook is saved
try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path(os.getcwd()).resolve()

# 2. Step UP out of 'notebooks' to the project root, then DOWN into 'data'
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

input_folder = project_root / "data" / "interim"
output_folder = project_root / "data" / "processed"
output_filename = "merged_features.csv"

# Create the processed folder automatically if it's missing
os.makedirs(output_folder, exist_ok=True)

# 3. Grab all CSV files inside that precise folder
search_path = os.path.join(input_folder, "*.csv")
csv_files = glob.glob(search_path)
csv_files = [f for f in csv_files if os.path.basename(f) != output_filename]

print(f"Project root directory: {project_root}")
print(f"Looking inside input path: {input_folder}")
print(f"Successfully found {len(csv_files)} file(s) to merge!")


Project root directory: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject
Looking inside input path: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim
Successfully found 10 file(s) to merge!


In [2]:
if len(csv_files) < 2:
    print(f"Found {len(csv_files)} file(s) in '{input_folder}'. You need at least 2 files to merge.")
    print("Ensure your VS Code terminal is open to the root folder containing 'data'.")
    exit()

# 2. Open the first file to establish the baseline
print(f"Reading base file: {csv_files[0]}")
master_df = pd.read_csv(csv_files[0])

# Automatically detect the names of your first two tracking columns
id_names = list(master_df.columns[:2])
print(f"Tracking keys detected: {id_names}")



Reading base file: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\additional_pair_features.csv
Tracking keys detected: ['user_a', 'user_b']


In [4]:
# 3. Loop through and merge the remaining files sideways
for file in csv_files[1:]:
    print(f"Merging: {file}")
    df_next = pd.read_csv(file)
    
    # Clean up column names to prevent case or space mismatches
    df_next.columns = df_next.columns.str.strip().str.lower()
    
    # Double check if both id_names are present in this specific file
    missing_keys = [key for key in id_names if key not in df_next.columns]
    if missing_keys:
        print(f"⚠️ Warning: Skipping {file} because it is missing tracking key columns: {missing_keys}")
        print(f"   Available columns in this file: {list(df_next.columns)}")
        continue  # Safely skip this broken file and move to the next one
        
    master_df = pd.merge(master_df, df_next, on=id_names, how='outer')

# 4. Save the wide feature matrix into the processed directory
destination_path = os.path.join(output_folder, output_filename)
master_df.to_csv(destination_path, index=False)

print(f"\nSuccess! Combined files aligned and saved to '{destination_path}'")
print(f"Total rows in final dataset: {len(master_df)}")


Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_days.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_totals.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pairwise_features.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_facebook_friend_counts.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_call_streaks.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_text_streaks.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_total_texts_sent.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\d